# 🚀 IDM-VTON HD Render Sunucusu (Google Colab L4/T4 GPU)

Bu notebook:
1. `transformers`, `diffusers`, `accelerate` ve `gradio` bağımlılık çakışmalarını temizleyip kararlı sürüm matrisini kurar.
2. `ModuleNotFoundError: No module named 'src'` ve `FileNotFoundError: app.py` yollarını dinamik algılama ve `sys.path` enjeksiyonuyla çözer.
3. Gerekli `ckpt` (OpenPose, HumanParsing, DensePose) modellerini otomatik senkronize eder.
4. Modelleri `torch.float16`, `device_map='cuda'` ve `low_cpu_mem_usage=True` ile GPU'ya yükleyerek Colab CPU RAM çökmesini (OOM) önler.
5. Gradio arayüzünü `share=True` ile güvenli genel erişime açar.

## 1. Bağımlılık Çakışmalarını Temizle ve Kararlı Sürüm Matrisini Kur

In [ ]:
!pip uninstall -y transformers diffusers accelerate huggingface_hub gradio pydantic tokenizers

!pip install -q \
    diffusers==0.25.0 \
    transformers==4.36.2 \
    accelerate==0.26.1 \
    gradio==4.44.1 \
    pydantic==2.10.6 \
    huggingface_hub==0.25.2 \
    peft==0.9.0 \
    einops==0.7.0 \
    onnxruntime \
    fvcore \
    cloudpickle \
    omegaconf \
    pycocotools \
    av \
    config==0.5.1

print("✅ Bağımlılık matrisi başarıyla kuruldu.")

## 2. Depoyu Klonla, app.py Konumunu Senkronize Et ve ckpt Modellerini Hazırla

In [ ]:
import os
import sys
import shutil

REPO_DIR = "/content/IDM-VTON"

if not os.path.exists(REPO_DIR):
    !git clone https://github.com/yisol/IDM-VTON.git {REPO_DIR}

# sys.path'e hem ana reponun hem de gradio_demo'nun eklenmesi
for p in [REPO_DIR, os.path.join(REPO_DIR, "gradio_demo")]:
    if p not in sys.path:
        sys.path.insert(0, p)

# GitHub klonunda app.py 'gradio_demo/app.py' altındadır. Root seviyesinde de erişilebilir yapalım:
gradio_app = os.path.join(REPO_DIR, "gradio_demo", "app.py")
root_app = os.path.join(REPO_DIR, "app.py")
if os.path.exists(gradio_app) and not os.path.exists(root_app):
    shutil.copy(gradio_app, root_app)
    print("✅ gradio_demo/app.py ana dizine senkronize edildi.")

# Gerekli yardımcı modellerin (OpenPose, HumanParsing, DensePose) ckpt kontrolü
ckpt_dir = os.path.join(REPO_DIR, "ckpt")
if not os.path.exists(ckpt_dir) or not os.listdir(ckpt_dir):
    print("📥 Yardımcı modeller (OpenPose, HumanParsing, DensePose) indiriliyor...")
    from huggingface_hub import snapshot_download
    snapshot_download("yisol/IDM-VTON", repo_type="space", allow_patterns=["ckpt/*"], local_dir=REPO_DIR)
    print("✅ Checkpoint modelleri hazır.")

%cd {REPO_DIR}
print(f"✅ Çalışma dizini: {os.getcwd()}")

## 3. Google Drive'ı Bağla ve Model Ağırlıklarını Hazırla

In [ ]:
import os
import shutil
from google.colab import drive

drive.mount("/content/drive")

DRIVE_WEIGHTS = "/content/drive/MyDrive/IDM-VTON-weights"
LOCAL_WEIGHTS = "/content/model_weights"

if os.path.exists(DRIVE_WEIGHTS):
    if not os.path.exists(LOCAL_WEIGHTS):
        print("[IO Engine] Model ağırlıkları Drive'dan yerel NVMe SSD'ye kopyalanıyor...")
        shutil.copytree(DRIVE_WEIGHTS, LOCAL_WEIGHTS)
        print("[IO Engine] Ağırlıklar yerel SSD önbelleğine alındı.")
    WEIGHTS_PATH = LOCAL_WEIGHTS
else:
    print(f"[Bilgi] '{DRIVE_WEIGHTS}' bulunamadı. Hugging Face Hub 'yisol/IDM-VTON' kullanılacak.")
    WEIGHTS_PATH = "yisol/IDM-VTON"

print(f"✅ Kullanılacak model yolu: {WEIGHTS_PATH}")

## 4. Kod Yamalarını Uygula, Düşük Bellek Modunu Etkinleştir ve Başlat

In [ ]:
import os
import sys
import re
import subprocess
import shutil
import torch

# 1. GPU Kontrolü
if not torch.cuda.is_available():
    print("❌ DİKKAT: GPU algılanamadı!")
    print("Lütfen üst menüden: Çalışma zamanı -> Çalışma zamanı türünü değiştir -> T4 GPU seçin.")
else:
    print(f"🔥 GPU Aktif: {torch.cuda.get_device_name(0)} (VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB)")

REPO_DIR = "/content/IDM-VTON"

# 2. Orijinal temiz repo durumuna sıfırla
subprocess.run(["git", "-C", REPO_DIR, "checkout", "--", "."], check=False)

gradio_app = os.path.join(REPO_DIR, "gradio_demo", "app.py")
root_app = os.path.join(REPO_DIR, "app.py")
if os.path.exists(gradio_app):
    shutil.copy(gradio_app, root_app)
    APP_PATH = root_app
elif os.path.exists(root_app):
    APP_PATH = root_app
else:
    raise FileNotFoundError(f"{REPO_DIR} altında app.py bulunamadı!")

for p in [REPO_DIR, os.path.join(REPO_DIR, "gradio_demo")]:
    if p not in sys.path:
        sys.path.insert(0, p)
os.chdir(REPO_DIR)

with open(APP_PATH, "r", encoding="utf-8") as f:
    content = f.read()

# 3. 'src' import hatasını önlemek için sys.path ve os.chdir enjeksiyonu
path_patch = f"""import sys, os
for _p in ["{REPO_DIR}", os.path.join("{REPO_DIR}", "gradio_demo")]:
    if _p not in sys.path:
        sys.path.insert(0, _p)
os.chdir("{REPO_DIR}")
"""
content = path_patch + content

# 4. base_path yönlendirmesi
content = re.sub(
    r"base_path\s*=\s*['\"][^'\"]+['\"]",
    f"base_path = '{WEIGHTS_PATH}'",
    content,
    count=1
)

# 5. KÖK ÇÖZÜM: TÜM modelleri (UNet, VAE, CLIP, GarmentNet) low_cpu_mem_usage=True ile yükle
# Bu adım Colab 12.7 GB CPU RAM çökmesini (OOM) %100 önler
content = re.sub(
    r"torch_dtype\s*=\s*torch\.float16",
    "torch_dtype=torch.float16, low_cpu_mem_usage=True",
    content
)

# 6. Modeller kurulur kurulmaz doğrudan GPU VRAM'ine aktar ve CPU RAM'i anında temizle
gpu_load_patch = """
pipe = pipe.to("cuda")
pipe.unet_encoder = UNet_Encoder.to("cuda")
openpose_model.preprocessor.body_estimation.model.to("cuda")
import gc; gc.collect(); torch.cuda.empty_cache()
print("🚀 TÜM MODELLER BAŞARIYLA GPU VRAM'İNE TAŞINDI! CPU RAM KORUMASI DEVREDE.")
"""
content = re.sub(
    r"pipe\.unet_encoder\s*=\s*UNet_Encoder",
    gpu_load_patch.strip(),
    content
)

# 7. Yükleme adımlarını canlı olarak görmek için progress print'leri ekle
content = content.replace(
    "unet.requires_grad_(False)",
    'unet.requires_grad_(False)\nprint("⏳ [1/5] UNet hazır. VAE ve CLIP kodlayıcıları yükleniyor...")'
)
content = content.replace(
    "vae.requires_grad_(False)",
    'vae.requires_grad_(False)\nprint("⏳ [2/5] VAE hazır. GarmentNet (UNet Encoder) yükleniyor...")'
)
content = content.replace(
    "UNet_Encoder.requires_grad_(False)",
    'UNet_Encoder.requires_grad_(False)\nprint("⏳ [3/5] GarmentNet hazır. TryonPipeline oluşturuluyor...")'
)

# 8. Spaces dekoratörünü devre dışı bırak
content = content.replace("@spaces.GPU", "# @spaces.GPU")

# 9. Gradio public linkini aç
content = re.sub(
    r"image_blocks\.launch\([^)]*\)",
    "image_blocks.launch(share=True, debug=True)",
    content
)

with open(APP_PATH, "w", encoding="utf-8") as f:
    f.write(content)

if os.path.exists(gradio_app):
    shutil.copy(root_app, gradio_app)

print("✅ app.py düşük bellek (low_cpu_mem_usage) ve GPU modunda yapılandırıldı. Başlatılıyor...")

%cd {REPO_DIR}
!python {APP_PATH}